In [ ]:
# Setup
!pip install -q tiktoken

import pandas as pd
import torch
import torch.nn as nn
import tiktoken

from sklearn.model_selection import train_test_split

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch version:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")


PyTorch version: 2.11.0+cu128
Device: cuda


In [2]:
# Task 1 - Load IMDB dataset

df = pd.read_csv("IMDB Dataset.csv")

print("Original dataset shape:", df.shape)

# Use more training examples so the RNN learns general sentiment words
# while keeping the notebook practical to run in Colab.
df = df.sample(n=10000, random_state=SEED).reset_index(drop=True)

print("Working dataset shape:", df.shape)
display(df.head())

print("\nSentiment distribution:")
print(df["sentiment"].value_counts())


Original dataset shape: (50000, 2)
Working dataset shape: (10000, 2)


,review,sentiment
0,I really liked this Summerslam due to the look...,positive
1,Not many television shows appeal to quite as m...,positive
2,The film quickly gets to a major chase scene w...,negative
3,Jane Austen would definitely approve of this o...,positive
4,Expectations were somewhat high for me when I ...,negative



Sentiment distribution:
sentiment
positive    5039
negative    4961
Name: count, dtype: int64


In [3]:
# Task 2 - positive = 1, negative = 0

df["sentiment"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

print(df["sentiment"].value_counts())


sentiment
1    5039
0    4961
Name: count, dtype: int64


In [4]:
# Task 3 - Train/test split

X = df["review"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 8000
Testing samples: 2000


In [5]:
# Task 4 - Tokenization

encoding = tiktoken.get_encoding("cl100k_base")

sample_review = X_train.iloc[0]
sample_tokens = encoding.encode(sample_review)

print("Review:")
print(sample_review)

print("\nToken IDs:")
print(sample_tokens)

print("\nNumber of tokens:", len(sample_tokens))


Review:
i totally disagree with the person who first commented about this movie. to me, this storyline is about a man with growth hormonal disorder trying his best to look for the love he is yearning for. Trying his very best to fit into the society who doesn't seemed to accept him.<br /><br />He then met someone who treats him with best friend's love and was shown the way to receiving the love he always wanted to have. the man who created all that happiness in returned received something important from the giant, which is family and spouse love.<br /><br />and the comedy, it's just part of making the audience feel some laughter bugs at certain part of the shows.<br /><br />and i must admit, it made me tear a little.

Token IDs:
[72, 12756, 29395, 449, 279, 1732, 889, 1176, 29786, 922, 420, 5818, 13, 311, 757, 11, 420, 51728, 374, 922, 264, 893, 449, 6650, 74218, 19823, 4560, 813, 1888, 311, 1427, 369, 279, 3021, 568, 374, 1060, 1251, 369, 13, 50958, 813, 1633, 1888, 311, 5052, 1139, 2

In [6]:
# Tokenize every review

X_train_tokens = [encoding.encode(review) for review in X_train]
X_test_tokens = [encoding.encode(review) for review in X_test]

print("Number of training reviews:", len(X_train_tokens))
print("Number of testing reviews:", len(X_test_tokens))
print("Tokens in first training review:", len(X_train_tokens[0]))


Number of training reviews: 8000
Number of testing reviews: 2000
Tokens in first training review: 154


In [7]:
# Task 5 - Fixed-length padding/truncation

MAX_LENGTH = 200

# Reserve an ID outside the tiktoken vocabulary for padding.
PAD_TOKEN_ID = encoding.n_vocab

def pad_or_truncate(sequences, max_length, pad_token_id):
    padded_sequences = []
    lengths = []

    for sequence in sequences:
        # Keep the number of real tokens after truncation.
        real_length = min(len(sequence), max_length)
        lengths.append(real_length)

        if len(sequence) > max_length:
            sequence = sequence[:max_length]
        else:
            sequence = sequence + [pad_token_id] * (max_length - len(sequence))

        padded_sequences.append(sequence)

    return padded_sequences, lengths

X_train_padded, X_train_lengths = pad_or_truncate(
    X_train_tokens, MAX_LENGTH, PAD_TOKEN_ID
)

X_test_padded, X_test_lengths = pad_or_truncate(
    X_test_tokens, MAX_LENGTH, PAD_TOKEN_ID
)

print("Training sequences:", len(X_train_padded))
print("Testing sequences:", len(X_test_padded))
print("Length of first training sequence:", len(X_train_padded[0]))
print("Real tokens in first training review:", X_train_lengths[0])
print("Padding token ID:", PAD_TOKEN_ID)


Training sequences: 8000
Testing sequences: 2000
Length of first training sequence: 200
Real tokens in first training review: 154
Padding token ID: 100277


In [8]:
# Task 6 - PyTorch tensors

X_train_tensor = torch.tensor(X_train_padded, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_padded, dtype=torch.long)

y_train_tensor = torch.tensor(
    y_train.to_numpy(), dtype=torch.float32
)
y_test_tensor = torch.tensor(
    y_test.to_numpy(), dtype=torch.float32
)

X_train_lengths_tensor = torch.tensor(X_train_lengths, dtype=torch.long)
X_test_lengths_tensor = torch.tensor(X_test_lengths, dtype=torch.long)

print("X_train shape:", X_train_tensor.shape)
print("X_test shape:", X_test_tensor.shape)
print("y_train shape:", y_train_tensor.shape)
print("y_test shape:", y_test_tensor.shape)


X_train shape: torch.Size([8000, 200])
X_test shape: torch.Size([2000, 200])
y_train shape: torch.Size([8000])
y_test shape: torch.Size([2000])


In [9]:
# Task 7 - Embedding + nn.RNN + Linear

class RNNTextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, pad_token_id):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_token_id
        )

        # Bidirectional nn.RNN is still an nn.RNN and lets the model
        # use information from both the beginning and end of a review.
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            nonlinearity="tanh",
            bidirectional=True
        )

        # Forward + backward hidden states are concatenated.
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        embedded = self.embedding(x)

        # h_n shape: (num_directions, batch, hidden_dim)
        _, h_n = self.rnn(embedded)

        # Last forward and last backward hidden states
        forward_hidden = h_n[0]
        backward_hidden = h_n[1]

        combined_hidden = torch.cat(
            (forward_hidden, backward_hidden), dim=1
        )

        return self.fc(combined_hidden)


In [10]:
# Model configuration

vocab_size = encoding.n_vocab + 1
embedding_dim = 128
hidden_dim = 96

model = RNNTextClassifier(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    pad_token_id=PAD_TOKEN_ID
)

print("Vocabulary size including PAD:", vocab_size)
print(model)

# Quick forward-pass check
sample_batch = X_train_tensor[:8]
sample_output = model(sample_batch)

print("Input shape:", sample_batch.shape)
print("Output shape:", sample_output.shape)


Vocabulary size including PAD: 100278
RNNTextClassifier(
  (embedding): Embedding(100278, 128, padding_idx=100277)
  (rnn): RNN(128, 96, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=192, out_features=1, bias=True)
)
Input shape: torch.Size([8, 200])
Output shape: torch.Size([8, 1])


In [11]:
# Task 8 - Training

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

X_train_tensor = X_train_tensor.to(device)
y_train_tensor = y_train_tensor.to(device)

criterion = nn.BCEWithLogitsLoss()

# A smaller learning rate and weight decay help prevent the severe
# overfitting seen with the previous 20-epoch run.
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0005,
    weight_decay=1e-5
)

EPOCHS = 12
BATCH_SIZE = 64

for epoch in range(EPOCHS):

    model.train()

    permutation = torch.randperm(X_train_tensor.size(0), device=device)
    total_loss = 0.0
    batch_count = 0

    for i in range(0, X_train_tensor.size(0), BATCH_SIZE):

        indices = permutation[i:i + BATCH_SIZE]

        batch_x = X_train_tensor[indices]
        batch_y = y_train_tensor[indices].view(-1, 1)

        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()
        batch_count += 1

    average_loss = total_loss / batch_count

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}], "
        f"Loss: {average_loss:.4f}"
    )


Epoch [1/12], Loss: 0.6953
Epoch [2/12], Loss: 0.6709
Epoch [3/12], Loss: 0.6419
Epoch [4/12], Loss: 0.5933
Epoch [5/12], Loss: 0.5311
Epoch [6/12], Loss: 0.4793
Epoch [7/12], Loss: 0.4307
Epoch [8/12], Loss: 0.3718
Epoch [9/12], Loss: 0.3329
Epoch [10/12], Loss: 0.2680
Epoch [11/12], Loss: 0.2225
Epoch [12/12], Loss: 0.1792


In [12]:
# Task 9 - Test-set evaluation

model.eval()

X_test_tensor = X_test_tensor.to(device)
y_test_tensor = y_test_tensor.to(device)

with torch.no_grad():
    outputs = model(X_test_tensor)
    probabilities = torch.sigmoid(outputs)
    predictions = (probabilities >= 0.5).long()

actual_labels = y_test_tensor.long().view(-1, 1)

correct = (predictions == actual_labels).sum().item()
total = actual_labels.size(0)
accuracy = correct / total * 100

print("Correct predictions:", correct)
print("Total test samples:", total)
print(f"Test Accuracy: {accuracy:.2f}%")


Correct predictions: 1303
Total test samples: 2000
Test Accuracy: 65.15%


In [13]:
# Task 10 - New review predictions

model.eval()

new_reviews = [
    "This movie was absolutely fantastic and I really enjoyed it.",
    "The movie was boring and a complete waste of time.",
    "I loved the story and the acting was excellent."
]

with torch.no_grad():

    for review in new_reviews:

        # Tokenize with the same tokenizer used for training
        tokens = encoding.encode(review)

        # Use the exact same fixed-length rule as Task 5
        padded_tokens, _ = pad_or_truncate(
            [tokens],
            MAX_LENGTH,
            PAD_TOKEN_ID
        )

        input_tensor = torch.tensor(
            padded_tokens,
            dtype=torch.long,
            device=device
        )

        output = model(input_tensor)
        probability = torch.sigmoid(output).item()

        if probability >= 0.5:
            sentiment = "Positive"
        else:
            sentiment = "Negative"

        print("Review:", review)
        print("Predicted Sentiment:", sentiment)
        print("Probability:", round(probability, 4))
        print()


Review: This movie was absolutely fantastic and I really enjoyed it.
Predicted Sentiment: Positive
Probability: 0.9766

Review: The movie was boring and a complete waste of time.
Predicted Sentiment: Negative
Probability: 0.1835

Review: I loved the story and the acting was excellent.
Predicted Sentiment: Positive
Probability: 0.984



In [14]:
# Activity 2 - Save Activity 1 results before creating the LSTM

# Keep a reference to the trained RNN from Activity 1 for comparison.
rnn_model = model
rnn_test_accuracy = accuracy

# Use the exact same three custom reviews from Activity 1.
new_reviews = [
    "This movie was absolutely fantastic and I really enjoyed it.",
    "The movie was boring and a complete waste of time.",
    "I loved the story and the acting was excellent."
]

print("Activity 1 RNN Test Accuracy:", f"{rnn_test_accuracy:.2f}%")


Activity 1 RNN Test Accuracy: 65.15%


In [15]:
# Activity 2 - Embedding + nn.LSTM + Linear

class LSTMTextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, pad_token_id):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_token_id
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        embedded = self.embedding(x)

        # h_n shape: (num_directions, batch, hidden_dim)
        _, (h_n, _) = self.lstm(embedded)

        # Last forward and backward hidden states
        forward_hidden = h_n[0]
        backward_hidden = h_n[1]

        combined_hidden = torch.cat(
            (forward_hidden, backward_hidden), dim=1
        )

        return self.fc(combined_hidden)


# Same model dimensions as Activity 1
lstm_model = LSTMTextClassifier(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    pad_token_id=PAD_TOKEN_ID
).to(device)

print(lstm_model)

# Quick forward-pass check
sample_batch = X_train_tensor[:8]
sample_output = lstm_model(sample_batch)

print("Input shape:", sample_batch.shape)
print("Output shape:", sample_output.shape)


LSTMTextClassifier(
  (embedding): Embedding(100278, 128, padding_idx=100277)
  (lstm): LSTM(128, 96, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=192, out_features=1, bias=True)
)
Input shape: torch.Size([8, 200])
Output shape: torch.Size([8, 1])


In [16]:
# Activity 2 - Train the LSTM with the same setup as Activity 1

lstm_criterion = nn.BCEWithLogitsLoss()

lstm_optimizer = torch.optim.Adam(
    lstm_model.parameters(),
    lr=0.0005,
    weight_decay=1e-5
)

EPOCHS_LSTM = 12
BATCH_SIZE_LSTM = 64

lstm_losses = []

for epoch in range(EPOCHS_LSTM):

    lstm_model.train()

    permutation = torch.randperm(X_train_tensor.size(0), device=device)
    total_loss = 0.0
    batch_count = 0

    for i in range(0, X_train_tensor.size(0), BATCH_SIZE_LSTM):

        indices = permutation[i:i + BATCH_SIZE_LSTM]

        batch_x = X_train_tensor[indices]
        batch_y = y_train_tensor[indices].view(-1, 1)

        outputs = lstm_model(batch_x)
        loss = lstm_criterion(outputs, batch_y)

        lstm_optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            lstm_model.parameters(),
            max_norm=1.0
        )

        lstm_optimizer.step()

        total_loss += loss.item()
        batch_count += 1

    average_loss = total_loss / batch_count
    lstm_losses.append(average_loss)

    print(
        f"Epoch [{epoch + 1}/{EPOCHS_LSTM}], "
        f"Loss: {average_loss:.4f}"
    )


Epoch [1/12], Loss: 0.6904
Epoch [2/12], Loss: 0.6624
Epoch [3/12], Loss: 0.5815
Epoch [4/12], Loss: 0.4925
Epoch [5/12], Loss: 0.4118
Epoch [6/12], Loss: 0.3488
Epoch [7/12], Loss: 0.2868
Epoch [8/12], Loss: 0.2377
Epoch [9/12], Loss: 0.1906
Epoch [10/12], Loss: 0.1518
Epoch [11/12], Loss: 0.1208
Epoch [12/12], Loss: 0.0849


In [17]:
# Activity 2 - Test-set evaluation

lstm_model.eval()

with torch.no_grad():
    lstm_outputs = lstm_model(X_test_tensor)
    lstm_probabilities = torch.sigmoid(lstm_outputs)
    lstm_predictions = (lstm_probabilities >= 0.5).long()

lstm_actual_labels = y_test_tensor.long().view(-1, 1)

lstm_correct = (lstm_predictions == lstm_actual_labels).sum().item()
lstm_total = lstm_actual_labels.size(0)
lstm_accuracy = lstm_correct / lstm_total * 100

print("Correct predictions:", lstm_correct)
print("Total test samples:", lstm_total)
print(f"LSTM Test Accuracy: {lstm_accuracy:.2f}%")


Correct predictions: 1506
Total test samples: 2000
LSTM Test Accuracy: 75.30%


In [18]:
# Activity 2 - LSTM predictions for the same Task 10 reviews

lstm_custom_predictions = []

lstm_model.eval()

with torch.no_grad():
    for review in new_reviews:

        tokens = encoding.encode(review)

        padded_tokens, _ = pad_or_truncate(
            [tokens],
            MAX_LENGTH,
            PAD_TOKEN_ID
        )

        input_tensor = torch.tensor(
            padded_tokens,
            dtype=torch.long,
            device=device
        )

        output = lstm_model(input_tensor)
        probability = torch.sigmoid(output).item()

        if probability >= 0.5:
            sentiment = "Positive"
        else:
            sentiment = "Negative"

        lstm_custom_predictions.append(sentiment)

        print("Review:", review)
        print("LSTM Predicted Sentiment:", sentiment)
        print("Probability:", round(probability, 4))
        print()


Review: This movie was absolutely fantastic and I really enjoyed it.
LSTM Predicted Sentiment: Positive
Probability: 0.9847

Review: The movie was boring and a complete waste of time.
LSTM Predicted Sentiment: Negative
Probability: 0.0069

Review: I loved the story and the acting was excellent.
LSTM Predicted Sentiment: Positive
Probability: 0.9893



In [19]:
# Activity 2 - Compare RNN and LSTM results

print("========== RNN vs LSTM Comparison ==========")
print(f"RNN Test Accuracy : {rnn_test_accuracy:.2f}%")
print(f"LSTM Test Accuracy: {lstm_accuracy:.2f}%")

if lstm_accuracy > rnn_test_accuracy:
    print("Higher test accuracy: LSTM")
elif lstm_accuracy < rnn_test_accuracy:
    print("Higher test accuracy: RNN")
else:
    print("Higher test accuracy: Both are equal")

print("\nCustom review comparison:")

# Get the RNN predictions using the exact same reviews and preprocessing.
rnn_custom_predictions = []
rnn_model.eval()

with torch.no_grad():
    for review in new_reviews:
        tokens = encoding.encode(review)
        padded_tokens, _ = pad_or_truncate(
            [tokens],
            MAX_LENGTH,
            PAD_TOKEN_ID
        )

        input_tensor = torch.tensor(
            padded_tokens,
            dtype=torch.long,
            device=device
        )

        output = rnn_model(input_tensor)
        probability = torch.sigmoid(output).item()
        sentiment = "Positive" if probability >= 0.5 else "Negative"
        rnn_custom_predictions.append(sentiment)

for i, review in enumerate(new_reviews):
    agreement = "Agree" if rnn_custom_predictions[i] == lstm_custom_predictions[i] else "Disagree"
    print(f"{i + 1}. RNN: {rnn_custom_predictions[i]} | LSTM: {lstm_custom_predictions[i]} | {agreement}")

print("\nTraining loss comparison:")
print(f"RNN final training loss : {average_loss:.4f}")
print(f"LSTM final training loss: {lstm_losses[-1]:.4f}")

if lstm_losses[-1] < average_loss:
    print("Lower final training loss: LSTM")
elif lstm_losses[-1] > average_loss:
    print("Lower final training loss: RNN")
else:
    print("Final training loss is equal")


========== RNN vs LSTM Comparison ==========
RNN Test Accuracy : 65.15%
LSTM Test Accuracy: 75.30%
Higher test accuracy: LSTM

Custom review comparison:
1. RNN: Positive | LSTM: Positive | Agree
2. RNN: Negative | LSTM: Negative | Agree
3. RNN: Positive | LSTM: Positive | Agree

Training loss comparison:
RNN final training loss : 0.0849
LSTM final training loss: 0.0849
Final training loss is equal
